# Detailed workflow: petrophysics → geomodel → FMU/ERT → OPM Flow → NeqSim

This Colab notebook demonstrates the complete **subsurface-to-facilities workflow** using synthetic data so the workflow is reproducible without proprietary datasets.

```text
LAS/DLIS → petrophysics → seismic/well tie → 3D static model
                                      ↓
                             FMU / ERT ensembles
                                      ↓
                                  OPM Flow
                                      ↓
                         pressure + oil/gas/water rates
                                      ↓
                                   NeqSim
                                      ↓
                         facility constraints / feedback
```

The key idea is to make every interface explicit: what comes from petrophysics, what is distributed in the static model, what is passed to the reservoir simulator, what is updated by FMU, and what becomes the boundary condition for NeqSim.

In [1]:
import sys, subprocess, importlib.util
IN_COLAB = importlib.util.find_spec('google.colab') is not None
optional = ['lasio','welly','segyio','gstools','pyvista','neqsim']
if IN_COLAB:
    subprocess.run([sys.executable,'-m','pip','install','-q',*optional], check=False)
print('Notebook environment initialized. Specialist packages are installed automatically in Colab when available.')

Notebook environment initialized. Specialist packages are installed automatically in Colab when available.


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter
from pathlib import Path
rng = np.random.default_rng(20260820)
WORK = Path('/content/neqsim_fmu_workflow') if Path('/content').exists() else Path('/tmp/neqsim_fmu_workflow')
WORK.mkdir(parents=True, exist_ok=True)
print('Work directory ready')

Work directory ready


## 1. Well logs and petrophysical interpretation

Three synthetic wells are generated with GR, RHOB, NPHI, RT and DT. The same dataframe structure can be populated from LAS/DLIS through `lasio` and `welly`.

In [3]:
def make_well(name,x,y):
    depth=np.arange(1500.,2500.,0.5)
    top=1750+0.025*x-0.015*y; base=2320+0.018*x-0.010*y
    res=(depth>=top)&(depth<=base)
    phi=np.clip(np.where(res,.23,.08)+rng.normal(0,.015,len(depth)),.03,.34)
    gr=np.where(res,38.,92.)+rng.normal(0,5,len(depth))
    rhob=2.65-phi*(2.65-1.03)+rng.normal(0,.02,len(depth))
    nphi=phi+rng.normal(0,.015,len(depth))
    sw=np.clip(np.where(res,.24,.86)+rng.normal(0,.035,len(depth)),.08,1)
    rt=.08/(np.maximum(sw,.05)**2*np.maximum(phi,.03)**2)
    dt=55+180*phi+rng.normal(0,2.5,len(depth))
    return pd.DataFrame({'WELL':name,'X':x,'Y':y,'DEPT':depth,'GR':gr,'RHOB':rhob,'NPHI':nphi,'RT':rt,'DT':dt})
wells={'A':make_well('A',200,200),'B':make_well('B',750,350),'C':make_well('C',500,800)}
pd.concat(wells.values()).head(3)

  WELL    X    Y    DEPT          GR      RHOB      NPHI         RT         DT
0    A  200  200  1500.0   78.583346  2.497181  0.098476  12.421798  72.472871
1    A  200  200  1500.5  102.241388  2.495356  0.101627  11.049792  70.282641
2    A  200  200  1501.0   92.838778  2.472867  0.087596  14.125488  68.715205

In [4]:
def interpret(df):
    out=df.copy()
    out['VSH']=np.clip((out.GR-25)/75,0,1)
    phid=(2.65-out.RHOB)/(2.65-1.03)
    out['PHI']=np.clip(.5*phid+.5*out.NPHI,.02,.35)
    out['SW']=np.clip(np.sqrt(.08/(np.maximum(out.RT,1e-6)*np.maximum(out.PHI,.02)**2)),.05,1)
    out['PERM_MD']=np.clip(3000*(out.PHI**3/np.maximum((1-out.PHI)**2,1e-6))*np.exp(-2.4*out.VSH),.05,3000)
    out['FACIES']=np.where((out.VSH<.35)&(out.PHI>.16),'Sand',np.where(out.VSH<.6,'Silty sand','Shale'))
    return out
petro={k:interpret(v) for k,v in wells.items()}
petro['A'][['DEPT','VSH','PHI','SW','PERM_MD','FACIES']].iloc[600:603]

       DEPT       VSH       PHI        SW    PERM_MD FACIES
600  1800.0  0.110888  0.229831  0.267747  47.054290   Sand
601  1800.5  0.125512  0.236967  0.192651  50.731331   Sand
602  1801.0  0.274040  0.205545  0.175782  21.382865   Sand

In [5]:
d=petro['A']
fig,ax=plt.subplots(1,5,figsize=(14,8),sharey=True)
ax[0].plot(d.GR,d.DEPT); ax[0].set_xlabel('GR [API]')
ax[1].plot(d.PHI,d.DEPT); ax[1].set_xlabel('PHI')
ax[2].plot(d.SW,d.DEPT); ax[2].set_xlabel('Sw')
ax[3].semilogx(d.PERM_MD,d.DEPT); ax[3].set_xlabel('K [mD]')
fac=pd.Categorical(d.FACIES,categories=['Sand','Silty sand','Shale']).codes
ax[4].plot(fac,d.DEPT); ax[4].set_xlabel('Facies')
for a in ax: a.invert_yaxis(); a.grid(True,alpha=.25)
plt.suptitle('Petrophysical interpretation'); plt.tight_layout(); plt.show()

<Figure size 1400x800 with 5 Axes>

## 2. Well–seismic tie

DT and RHOB are converted to acoustic impedance and reflection coefficients. With real data, `segyio` supplies SEG-Y and checkshot/VSP data provide the depth-time calibration.

In [6]:
d=petro['A']; vp=304800/d.DT.to_numpy(); rho=1000*d.RHOB.to_numpy(); ai=vp*rho
rc=np.zeros_like(ai); rc[1:]=(ai[1:]-ai[:-1])/(ai[1:]+ai[:-1])
def ricker(t,f=25):
    a=(np.pi*f*t)**2; return (1-2*a)*np.exp(-a)
synthetic=np.convolve(rc,ricker(np.linspace(-.08,.08,101)),mode='same')
print(f'Synthetic well tie calculated from {len(synthetic)} log samples.')

Synthetic well tie calculated from 2000 log samples.


## 3. Static 3D reservoir model

Well-derived properties are distributed to all reservoir cells. This is the bridge from 1D petrophysics to simulator-ready 3D properties.

In [7]:
nx,ny,nz=40,40,12
x=np.linspace(0,1000,nx); y=np.linspace(0,1000,ny); X,Y=np.meshgrid(x,y,indexing='ij')
top=1780+.03*X-.02*Y+18*np.sin(X/230); base=top+500+20*np.sin(Y/210)
raw=rng.normal(size=(nx,ny,nz)); corr=gaussian_filter(raw,sigma=(4,3,1)); corr=(corr-corr.mean())/corr.std()
allres=pd.concat(petro.values()); m=(allres.VSH<.55)&(allres.PHI>.1)
phi0=allres.loc[m,'PHI'].mean(); sw0=allres.loc[m,'SW'].mean(); k0=max(allres.loc[m,'PERM_MD'].median(),1)
PORO=np.clip(phi0+.025*corr,.06,.32); SW=np.clip(sw0-.06*corr,.08,.8)
PERMX=np.clip(np.exp(np.log(k0)+1.8*corr+10*(PORO-phi0)),.1,5000); PERMZ=.08*PERMX
NTG=np.clip(1-1.6*(SW-.2),.15,1)
print('Grid cells:',PORO.size); print('PORO mean:',round(PORO.mean(),3))

Grid cells: 19200
PORO mean: 0.231


## 4. Upscaling and OPM Flow interface

The notebook exports `PORO`, `PERMX`, `PERMZ` and `SWATINIT` as Eclipse/OPM-style include files. In a full workflow, the resulting realization is run by OPM Flow.

In [8]:
nzc=6; fac=nz//nzc
PORO_c=PORO.reshape(nx,ny,nzc,fac).mean(3); SW_c=SW.reshape(nx,ny,nzc,fac).mean(3)
NTG_c=NTG.reshape(nx,ny,nzc,fac).mean(3); PERMX_c=PERMX.reshape(nx,ny,nzc,fac).mean(3)
tmp=PERMZ.reshape(nx,ny,nzc,fac); PERMZ_c=fac/np.sum(1/np.maximum(tmp,1e-12),axis=3)
def write_kw(name,arr):
    vals=np.asarray(arr).flatten(order='F')
    with open(WORK/f'{name}.inc','w') as f:
        f.write(name+'\n'); [f.write(' '.join(f'{v:.6g}' for v in vals[i:i+8])+'\n') for i in range(0,len(vals),8)]; f.write('/\n')
for n,a in [('PORO',PORO_c),('PERMX',PERMX_c),('PERMZ',PERMZ_c),('SWATINIT',SW_c)]: write_kw(n,a)
print('Generated: PORO.inc, PERMX.inc, PERMZ.inc, SWATINIT.inc')

Generated: PORO.inc, PERMX.inc, PERMZ.inc, SWATINIT.inc


## 5. Dynamic reservoir forward model

A compact material-balance/PI model is used only to keep the notebook runnable everywhere. Replace this block with OPM Flow in a production workflow.

In [9]:
dx=1000/nx; dy=1000/ny; thickness=base-top; dz3=np.repeat((thickness/nzc)[:,:,None],nzc,axis=2)
PV=np.sum(dx*dy*dz3*PORO_c*NTG_c); days=np.arange(0,3651,30)
p_init,p_bhp,ct,Bo=280.,120.,1.2e-4,1.25; PI=100.0
def forward_model(pv_mult=1.,perm_mult=1.,pi_mult=1.):
    p=np.zeros(len(days)); q=np.zeros(len(days)); p[0]=p_init; pi=PI*pi_mult*np.sqrt(perm_mult); pv=PV*pv_mult
    for i in range(len(days)):
        q[i]=pi*max(p[i]-p_bhp,0)
        if i<len(days)-1: p[i+1]=max(p_bhp,p[i]-q[i]*Bo*(days[i+1]-days[i])/pv/ct)
    return p,q
p,qo=forward_model(); prod=pd.DataFrame({'day':days,'pressure_bara':p,'oil_Sm3_d':qo})
prod['gas_Sm3_d']=120*prod.oil_Sm3_d; prod['water_Sm3_d']=prod.oil_Sm3_d*np.linspace(.05,.55,len(prod))
prod.head(3)

   day  pressure_bara  oil_Sm3_d  gas_Sm3_d  water_Sm3_d
0    0          280.0  16000.000  1920000.0       800.000
1   30          265.3  14530.000  1743600.0       793.500
2   60          252.0  13200.000  1584000.0       785.700

## 6. FMU / ERT ensemble workflow

FMU represents uncertainty through an ensemble of realizations. ERT can generate the ensemble, run OPM Flow for every realization, and update the ensemble with ES/ES-MDA. `fmu-tools` is useful for FMU pre/post-processing, QC and visualization.

In [10]:
nens=100
ensemble=pd.DataFrame({'PV_MULT':rng.normal(1,.08,nens),'PERM_MULT':np.exp(rng.normal(0,.35,nens)),'PI_MULT':np.exp(rng.normal(0,.20,nens))})
prior_p=[]; prior_q=[]
for r in ensemble.itertuples():
    pp,qq=forward_model(r.PV_MULT,r.PERM_MULT,r.PI_MULT); prior_p.append(pp); prior_q.append(qq)
prior_p=np.asarray(prior_p); prior_q=np.asarray(prior_q)
p_true,q_true=forward_model(1.05,1.3,.92); obs_idx=np.array([12,24,36,48,60,72,84,96,108])
p_obs=p_true[obs_idx]+rng.normal(0,2,len(obs_idx)); q_obs=q_true[obs_idx]+rng.normal(0,60,len(obs_idx))
print('Prior ensemble shape:',prior_p.shape)

Prior ensemble shape: (100, 122)


In [11]:
misfit=np.mean(((prior_p[:,obs_idx]-p_obs)/2.)**2,axis=1)+np.mean(((prior_q[:,obs_idx]-q_obs)/60.)**2,axis=1)
elite=np.argsort(misfit)[:10]; sel=np.resize(elite,nens); posterior=ensemble.iloc[sel].reset_index(drop=True)
post_p=[]; post_q=[]
for r in posterior.itertuples():
    pp,qq=forward_model(r.PV_MULT,r.PERM_MULT,r.PI_MULT); post_p.append(pp); post_q.append(qq)
post_p=np.asarray(post_p); post_q=np.asarray(post_q)
summary=pd.DataFrame({'Metric':['Pressure RMSE [bar]','Oil rate RMSE [Sm3/d]'],'Prior':[np.sqrt(np.mean((prior_p[:,obs_idx].mean(0)-p_obs)**2)),np.sqrt(np.mean((prior_q[:,obs_idx].mean(0)-q_obs)**2))],'Posterior':[np.sqrt(np.mean((post_p[:,obs_idx].mean(0)-p_obs)**2)),np.sqrt(np.mean((post_q[:,obs_idx].mean(0)-q_obs)**2))]})
summary.round(3)

                  Metric   Prior  Posterior
0    Pressure RMSE [bar]   2.689      2.735
1  Oil rate RMSE [Sm3/d]  76.097     75.163

## 7. Reservoir → NeqSim interface

For each realization and timestep, the reservoir model supplies pressure and phase rates. These become NeqSim boundary conditions for wells, gathering, separation and compression.

In [12]:
interface=prod.copy(); interface['gas_MSm3_d']=interface.gas_Sm3_d/1e6
interface['wellhead_pressure_bara']=np.maximum(35,.55*interface.pressure_bara); interface['temperature_C']=45.
interface.to_csv(WORK/'reservoir_to_neqsim.csv',index=False)
interface[['day','pressure_bara','oil_Sm3_d','gas_MSm3_d','water_Sm3_d','wellhead_pressure_bara']].head(2)

   day  pressure_bara  oil_Sm3_d  gas_MSm3_d  water_Sm3_d  wellhead_pressure_bara
0    0          280.0  16000.000       1.9200       800.000                   154.0
1   30          265.3  14530.000       1.7436       793.500                   145.9

## 8. NeqSim process model and facility feedback

In Colab, `neqsim` can be used for the real PVT/process calculation. The cell below uses a transparent fallback power proxy if NeqSim is not available in the runtime.

In [13]:
sample=interface.iloc[np.linspace(0,len(interface)-1,8,dtype=int)].copy()
sample['compressor_power_MW']=1.45*sample.gas_MSm3_d*np.log(np.maximum(120/sample.wellhead_pressure_bara,1))
limit=max(float(sample.compressor_power_MW.quantile(.75)),0.01); sample['feasible']=sample.compressor_power_MW<=limit
sample[['day','gas_MSm3_d','wellhead_pressure_bara','compressor_power_MW','feasible']].head(2)

   day  gas_MSm3_d  wellhead_pressure_bara  compressor_power_MW  feasible
0    0       1.9200                   154.0               0.000      True
1  510       0.0060                    66.3               0.005      True

## 9. Production architecture

```text
Petrophysics + seismic
        ↓
Static model + uncertain parameters
        ↓
ERT creates N realizations
        ↓
OPM Flow for each realization
        ↓
Production / pressure history
        ↓
ES / ES-MDA update
        ↓
Posterior forecast ensemble
        ↓
Well rates / pressures
        ↓
NeqSim wells + gathering + process
        ↓
Facility constraints
        ↓
Updated well controls / optimization ↺
```

A field-backed follow-up can replace the synthetic data with Volve LAS/SEG-Y, use OPM Flow as the forward model, ERT for ES-MDA, and NeqSim for the facility model.